In [ ]:
# Load a expression from json
from asmblr.base import BaseNode
import os
import json
from migumi.utils.converter import fix_format, get_expr_and_state, fix_expr_dict
import migumi.shader
from migumi.shader.compiler import compile_set
from migumi.shader.compile_multipass import compile_set_multipass

from geolipi.torch_compute.sketcher import Sketcher
res = 1024
sketcher_2d = Sketcher(n_dims=2, resolution=res, device='cuda')
sketcher_3d = Sketcher(n_dims=3, resolution=256, device='cuda')
uniforms = {}
# resolved_expr = map_state_to_expr(expr_dict, sketcher_3d, state_map, uniforms)
design_dir = "/users/aganesh8/data/aganesh8/projects/kigumi/migumi-dataset/joints/"

# cur_design = "three_cubes.json"
cur_design = "CJ_AKT/polyline_files/base.json"
# cur_design ="CJ_DT.json"
cur_design_file = os.path.join(design_dir, cur_design)

design_name = cur_design.split(".")[0]


In [ ]:

data =  json.load(open(cur_design_file, "r"))
main_key = list(data['moduleList'].keys())[0]
data = data['moduleList'][main_key]

corrected_data = fix_format(data)
expressions = BaseNode.from_dict(data)
if not isinstance(expressions, list):
    expressions = [expressions]
expr_dict, state_map = get_expr_and_state(expressions)
expr_dict = fix_expr_dict(expr_dict, mode="v4", add_bounding=False)

In [ ]:

import geolipi.symbolic as gls
import sysl.symbolic as csls
from sysl.shader.shader_module import SMMap
from IPython.display import display, HTML
from sysl.shader_vis.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from sysl.shader_vis.generate_shader_html import create_multibuffer_shader_html

settings = {
    "render_mode": "v4",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": True,
        "_AA": 1,
        "_RAYCAST_MAX_STEPS": 200,
        "resolution": (512, 512),
        "outline_nhbd": 1,

    },
    "set_to_ubo": False,
    "export_params": False,
    # "target": "ShaderToy",
    # "convert_uniforms_to_constants": True,

}
# shader_code, uniforms, textures = compile_set(expr_dict, state_map, settings=settings)
# with open("/users/aganesh8/data/aganesh8/projects/kigumi/site/shader.frag", "w") as f:
#     f.write(shader_code)
# html_code = create_shader_html(shader_code, uniforms, textures, show_controls=True, backend="twgl")
# with open("/users/aganesh8/data/aganesh8/projects/kigumi/site/test.html", "w") as f:
#     f.write(html_code)
# # To visualize inline in jupyter notebook:
# jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))

all_shader_bundles = compile_set_multipass(expr_dict, state_map, settings=settings,
     post_process_shader=["part_outline_nobg"])
output_html = create_multibuffer_shader_html(all_shader_bundles, show_controls=True, backend="twgl")
with open("/users/aganesh8/data/aganesh8/projects/kigumi/site/test.html", "w") as f:
    f.write(output_html)
# To visualize inline in jupyter notebook:
# jupy_wrapper_html = make_jupyter_compatible_html(output_html)
# display(HTML(jupy_wrapper_html))


In [ ]:
for ind, shader_bundle in enumerate(all_shader_bundles):
    code = shader_bundle["shader_code"]
    with open(f"/users/aganesh8/data/aganesh8/projects/kigumi/site/shader_{ind}.frag", "w") as f:
        f.write(code)


In [ ]:
# Add the wood texture from Maria's previous work. 
